# LangGraph Tutorials

This notebook demonstrates:
1. A simple linear LangGraph workflow
2. A validation/self-correction loop
3. A ReAct-style agent with tool calling
4. Streaming graph execution

The code uses the current LangGraph message-state pattern.


In [1]:
# Install required packages
!pip install -U langgraph langchain langchain-openai python-dotenv -q


### 2. Set OpenAI API Key

Create a `.env` file in the same folder as this notebook:

```text
OPENAI_API_KEY=your_api_key_here
```

Do not hard-code your API key in the notebook.


In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(
        "OPENAI_API_KEY is not set. Add it to your .env file "
        "or set it in your environment before running the notebook."
    )


### 3. Imports & LLM Setup

In [3]:
from typing import Annotated, Optional
from typing_extensions import TypedDict

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition

from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import BaseMessage, HumanMessage

# Initialize LLM
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
)


c:\Users\win10\Desktop\Agentic AI Course\agentic_ai_course_batch_2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### 4. Define State and Nodes

`StateGraph` requires a state schema. `add_messages` is used as the reducer so nodes can return only the new messages instead of manually rebuilding the entire message history.


In [4]:
class MyState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    error: Optional[str]
    next: str
    validation_attempts: int


def llm_node(state: MyState):
    response = llm.invoke(state["messages"])

    return {
        "messages": [response],
        "error": None,
        "next": "",
        "validation_attempts": state.get("validation_attempts", 0) + 1,
    }


def validator_node(state: MyState):
    messages = state["messages"]
    last_content = messages[-1].content if messages else ""

    if isinstance(last_content, str) and "error" in last_content.lower():
        return {
            "error": "Validation failed",
            "next": "llm",
        }

    return {
        "error": None,
        "next": "end",
    }


### 5. Simple Linear Graph

The graph is:

```text
START → LLM → Validator → END
```


In [5]:
workflow = StateGraph(MyState)

workflow.add_node("llm", llm_node)
workflow.add_node("validator", validator_node)

workflow.add_edge(START, "llm")
workflow.add_edge("llm", "validator")
workflow.add_edge("validator", END)

basic_app = workflow.compile()

# Test
initial_state = {
    "messages": [
        HumanMessage(content="Hello! Tell me about yourself.")
    ],
    "error": None,
    "next": "",
}

result = basic_app.invoke(initial_state)

print("Final Response:")
print(result["messages"][-1].content)


Final Response:
Hello! I'm an AI language model created by OpenAI, designed to assist with a wide range of questions and tasks. I can provide information, answer questions, help with writing, and engage in conversation on various topics. My knowledge is based on a diverse set of texts up until October 2023, and I'm here to help you with whatever you need! What would you like to know or discuss?


### 6. Graph with Loop (Self-Correction)

This version demonstrates conditional routing:

```text
                 ┌──────────────┐
                 │              │
START → LLM → Validator ────────┘
                 │
                 └────────────→ END
```

If validation fails, execution goes back to the LLM.


In [7]:
MAX_VALIDATION_ATTEMPTS = 3


def route_after_validator(state: MyState):
    attempts = state.get("validation_attempts", 0)

    # Stop if validation succeeds
    if not state.get("error"):
        return END

    # Stop if maximum attempts reached
    if attempts >= MAX_VALIDATION_ATTEMPTS:
        print(f"Maximum validation attempts ({MAX_VALIDATION_ATTEMPTS}) reached.")
        return END

    # Retry LLM
    return "llm"


workflow_cycle = StateGraph(MyState)

workflow_cycle.add_node("llm", llm_node)
workflow_cycle.add_node("validator", validator_node)

workflow_cycle.add_edge(START, "llm")
workflow_cycle.add_edge("llm", "validator")

workflow_cycle.add_conditional_edges(
    "validator",
    route_after_validator,
    {
        "llm": "llm",
        END: END,
    },
)

cycle_app = workflow_cycle.compile()

result = cycle_app.invoke({
    "messages": [
        HumanMessage(
            content="Generate a structured list of three AI concepts."
        )
    ],
    "error": None,
    "next": "",
    "validation_attempts": 0,
})

print("Final Response:")
print(result["messages"][-1].content)

print(
    "Validation attempts:",
    result["validation_attempts"]
)


Maximum validation attempts (3) reached.
Final Response:
Sure! Here’s a structured list of three AI concepts:

### 1. Machine Learning (ML)
   - **Definition**: A subset of AI that enables systems to learn from data and improve their performance over time without being explicitly programmed.
   - **Types**:
     - **Supervised Learning**: Learning from labeled data to make predictions.
     - **Unsupervised Learning**: Finding patterns in unlabeled data.
     - **Reinforcement Learning**: Learning through trial and error to maximize a reward.
   - **Applications**:
     - Image and speech recognition
     - Recommendation systems
     - Fraud detection

### 2. Natural Language Processing (NLP)
   - **Definition**: A field of AI that focuses on the interaction between computers and humans through natural language.
   - **Key Components**:
     - **Text Analysis**: Understanding and processing human language in text form.
     - **Speech Recognition**: Converting spoken language into tex

### 7. ReAct Agent

The important part of a tool-calling graph is:

```text
START → Agent → Tools → Agent → ... → END
```

When the model requests a tool, `ToolNode` executes it and creates the required `ToolMessage`. This prevents the OpenAI error that occurs when an assistant `tool_call` is not followed by a matching tool response.


In [8]:
@tool
def search_tool(query: str) -> str:
    """Search for information about a topic.

    This is a dummy search tool for demonstrating LangGraph tool calling.
    """
    return (
        f"Search results for '{query}': "
        f"Found relevant information about {query}."
    )


tools = [search_tool]

# Bind the tools to the LLM so it can decide when to call them.
llm_with_tools = llm.bind_tools(tools)


def agent_node(state: MyState):
    """Agent node that can decide whether to call a tool."""
    response = llm_with_tools.invoke(state["messages"])

    return {
        "messages": [response],
        "error": None,
        "next": "",
    }


# ToolNode executes tool_calls and appends ToolMessage objects.
tool_node = ToolNode(tools)


# Build ReAct graph
react_graph = StateGraph(MyState)

react_graph.add_node("agent", agent_node)
react_graph.add_node("tools", tool_node)

react_graph.add_edge(START, "agent")

# tools_condition routes:
#   tool call present -> "tools"
#   no tool call       -> END
react_graph.add_conditional_edges(
    "agent",
    tools_condition,
    {
        "tools": "tools",
        END: END,
    },
)

# After a tool executes, send the result back to the agent.
react_graph.add_edge("tools", "agent")

react_app = react_graph.compile()


### 8. Test the ReAct Agent

In [9]:
final_result = react_app.invoke({
    "messages": [
        HumanMessage(content="Search for what is LangGraph")
    ],
    "error": None,
    "next": "",
})

print("Final Answer:")
print(final_result["messages"][-1].content)


Final Answer:
LangGraph is a platform designed to facilitate the development and deployment of applications that utilize language models. It provides tools and frameworks that enable developers to create, manage, and optimize language-based applications efficiently. LangGraph focuses on enhancing the interaction between users and language models, making it easier to integrate natural language processing capabilities into various applications.


### 9. Stream Output

`stream_mode="values"` shows the state after each graph step. Tool calls may have empty textual content, so the helper below prints the message type and useful content.


In [10]:
print("Streaming output:\n")

for state in react_app.stream(
    {
        "messages": [
            HumanMessage(content="What can LangGraph do?")
        ],
        "error": None,
        "next": "",
    },
    stream_mode="values",
):
    messages = state.get("messages", [])

    if messages:
        last_message = messages[-1]

        print(f"Message type: {type(last_message).__name__}")

        if getattr(last_message, "tool_calls", None):
            print("Tool calls:", last_message.tool_calls)

        if getattr(last_message, "content", None):
            print("Content:", last_message.content)

        print("-" * 60)


Streaming output:

Message type: HumanMessage
Content: What can LangGraph do?
------------------------------------------------------------
Message type: AIMessage
Tool calls: [{'name': 'search_tool', 'args': {'query': 'What can LangGraph do?'}, 'id': 'call_4BoDjQjPJu5CKVOMNKRs3z1O', 'type': 'tool_call'}]
------------------------------------------------------------
Message type: ToolMessage
Content: Search results for 'What can LangGraph do?': Found relevant information about What can LangGraph do?.
------------------------------------------------------------
Message type: AIMessage
Content: LangGraph is a versatile tool that can assist with various tasks, including:

1. **Natural Language Processing**: It can analyze and understand human language, enabling applications like sentiment analysis, text summarization, and language translation.

2. **Data Retrieval**: LangGraph can search for and retrieve information from various sources, making it useful for research and information gatheri